In [ ]:
import os, math, time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("Device:", device)

Device: cuda


In [ ]:
data_path = '/kaggle/working/data'
os.makedirs(data_path, exist_ok=True)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
# train_transform = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(10),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
# ])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_set = datasets.OxfordIIITPet(root=data_path, split='trainval',
    target_types='category', download=True, transform=train_transform)
test_set = datasets.OxfordIIITPet(root=data_path, split='test',
    target_types='category', download=True, transform=test_transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_set)} | Test: {len(test_set)} | Classes: 37")

100%|██████████| 792M/792M [00:46<00:00, 17.2MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 7.51MB/s]


Train: 3680 | Test: 3669 | Classes: 37


In [ ]:
class LoRAConv2d(nn.Module):

    def __init__(self, original_conv: nn.Conv2d, rank: int = 4, alpha: float = 8.0):
        super().__init__()

        self.original_conv = original_conv


        self.rank = rank
        self.scale = alpha / rank # paper found this is optimal

        out_channels, in_channels, kernel_size, _ = original_conv.weight.shape

        self.out_channels = out_channels
        self.in_channels = in_channels
        self.kernel_size = kernel_size

        self.lora_A = nn.Parameter(
            torch.empty(rank, in_channels * kernel_size)
        )
        self.lora_B = nn.Parameter(
            torch.zeros(out_channels * kernel_size, rank)
        )

        # from the paper
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        # freeze original convolution
        for param in self.original_conv.parameters():
            param.requires_grad = False

    def forward(self, x):
        # shape after multiplication:
        # (out_channels * kernel_size, in_channels * kernel_size)
        lora_update = self.lora_B @ self.lora_A

        lora_update = lora_update.view(
            self.out_channels,
            self.kernel_size,
            self.in_channels,
            self.kernel_size
        )

        lora_update = lora_update.permute(0, 2, 1, 3)

        # add lora update to the original weights we froze
        adapted_weight = self.original_conv.weight + self.scale * lora_update

        return nn.functional.conv2d(
            x,
            adapted_weight,
            self.original_conv.bias,
            stride=self.original_conv.stride,
            padding=self.original_conv.padding,
            dilation=self.original_conv.dilation,
            groups=self.original_conv.groups
        )

In [ ]:
def freeze_bn_stats(model):
    # explicitly freeze BN layer, not just gamma/beta but also runnign mean and variance
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum / len(loader), 100 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        out = model(imgs)
        loss_sum += criterion(out, labels).item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum / len(loader), 100 * correct / total

def count_trainable(model):
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    a = sum(p.numel() for p in model.parameters())
    return t, a

In [ ]:
def train_one_epoch_with_frozen_bn(model, loader, optimizer, criterion):
    model.train()
    freeze_bn_stats(model)

    loss_sum, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return loss_sum / len(loader), 100 * correct / total

In [ ]:
def build_lora_c_resnet101(num_classes=37, rank=4, alpha=None, unfreeze_bn=False):

    if alpha is None:
        alpha = 2.0 * rank

    model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    # freeze all params
    for p in model.parameters():
        p.requires_grad = False


    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for name, module in list(layer.named_modules()):
            if isinstance(module, nn.Conv2d):
                parts = name.split('.')
                parent = layer
                for part in parts[:-1]:
                    parent = getattr(parent, part)
                setattr(parent, parts[-1], LoRAConv2d(module, rank=rank, alpha=alpha))

    for p in model.conv1.parameters():
        p.requires_grad = True
    for p in model.fc.parameters():
        p.requires_grad = True


    # optinally unfreeze BN
    if unfreeze_bn:
        for m in model.modules():
            if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                for p in m.parameters():
                    p.requires_grad = True

    trainable, total = count_trainable(model)
    print(f"LoRA-C ResNet-101 (r={rank}, α={alpha}) | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model

In [ ]:
def build_finetune_resnet101(num_classes=37):
    model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    trainable, total = count_trainable(model)
    print(f"Full fine-tune ResNet-101 | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model

def build_linear_probe_resnet101(num_classes=37):
    model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    trainable, total = count_trainable(model)
    print(f"Linear probe ResNet-101 | Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model

# Corrupted data

created corrupted data experiments with greyscale, guassian noise, blur, brightness, contrast

In [ ]:
import torch
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
from PIL import ImageFilter, ImageEnhance

class CorruptedDataset(Dataset):
    def __init__(self, base_dataset, corruption_fn, transform):
        self.base_dataset = base_dataset
        self.corruption_fn = corruption_fn
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, label = self.base_dataset._images[idx], self.base_dataset._labels[idx]
        from PIL import Image
        img = Image.open(img).convert('RGB')
        if self.corruption_fn is not None:
            img = self.corruption_fn(img)
        img = self.transform(img)
        return img, label



def grayscale_corrupt(img):
    return img.convert('L').convert('RGB')

def gaussian_noise_corrupt(img, severity=0.1):
    arr = np.array(img).astype(np.float32) / 255.0
    arr += np.random.normal(0, severity, arr.shape)
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    from PIL import Image
    return Image.fromarray(arr)

def blur_corrupt(img, radius=3):
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def brightness_corrupt(img, factor=0.4):
    return ImageEnhance.Brightness(img).enhance(factor)

def contrast_corrupt(img, factor=0.3):
    return ImageEnhance.Contrast(img).enhance(factor)

raw_test = datasets.OxfordIIITPet(root=data_path, split='test',
    target_types='category', download=False)

corruptions = {
    'clean':      None,
    'grayscale':  grayscale_corrupt,
    'noise':      gaussian_noise_corrupt,
    'blur':       blur_corrupt,
    'brightness': brightness_corrupt,
    'contrast':   contrast_corrupt,
}

corrupt_loaders = {}
for cname, cfn in corruptions.items():
    ds = CorruptedDataset(raw_test, cfn, test_transform)
    corrupt_loaders[cname] = DataLoader(ds, batch_size=64, shuffle=False,
                                         num_workers=2, pin_memory=True)
    print(f"  {cname}: {len(ds)} images")

  clean: 3669 images
  grayscale: 3669 images
  noise: 3669 images
  blur: 3669 images
  brightness: 3669 images
  contrast: 3669 images


In [ ]:
import pandas as pd

criterion = nn.CrossEntropyLoss()
N_EPOCHS = 15

configs = [
    ("Linear Probe",   build_linear_probe_resnet101, 0.01, 5e-4, 0.9),
    ("Full Fine-tune",  build_finetune_resnet101,    0.01, 5e-4, 0.9),
    ("LoRA-C (r=4)",    build_lora_c_resnet101,      0.01, 5e-4, 0.9),
]

robustness_results = []

for name, builder, lr, wd, mom in configs:
    print(f"\n{'='*60}\n  Training: {name}\n{'='*60}")
    model = builder().to(device)

    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, momentum=mom, weight_decay=wd
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    for ep in range(1, N_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        te_loss, te_acc = evaluate(model, test_loader, criterion)
        scheduler.step()
        print(f"  Ep {ep:2d}/{N_EPOCHS} | Train {tr_acc:5.1f}% | Test {te_acc:5.1f}% | {time.time()-t0:.0f}s")

    print(f"\n evaluation: {name} ---")
    for cname, cloader in corrupt_loaders.items():
        _, acc = evaluate(model, cloader, criterion)
        robustness_results.append({
            'method': name, 'corruption': cname, 'accuracy': acc
        })
        print(f"    {cname:15s}: {acc:.1f}%")

    del model
    torch.cuda.empty_cache()


rob_df = pd.DataFrame(robustness_results)
pivot = rob_df.pivot(index='method', columns='corruption', values='accuracy')
pivot['mean_corrupt'] = pivot[[c for c in pivot.columns if c != 'clean']].mean(axis=1)
pivot = pivot.round(1)
print("\n" + "="*80)
print("  ROBUSTNESS COMPARISON")
print("="*80)
print(pivot.to_string())

# Limited data experiment

In [ ]:
# Limited data experiment, r=4, lr=0.01 for all, unfrozen BN

from torch.utils.data import Subset
import numpy as np
import pandas as pd

N_EPOCHS = 15
criterion = nn.CrossEntropyLoss()

def get_stratified_subset(dataset, fraction, seed=42):
    np.random.seed(seed)
    targets = np.array([dataset[i][1] for i in range(len(dataset))])
    indices = []
    for c in np.unique(targets):
        c_idx = np.where(targets == c)[0]
        n_keep = max(1, int(len(c_idx) * fraction))
        indices.extend(np.random.choice(c_idx, n_keep, replace=False))
    return Subset(dataset, indices)

fractions = [0.1,0.25, 0.5, 1.0]
limited_results = []

for frac in fractions:
    subset = get_stratified_subset(train_set, frac)
    sub_loader = DataLoader(subset, batch_size=64, shuffle=True,
                            num_workers=2, pin_memory=True)
    n_images = len(subset)
    print(f"\n{'='*60}")
    print(f"  Data: {frac*100:.0f}% ({n_images} images)")
    print(f"{'='*60}")

    configs = [
        ("Linear Probe",   build_linear_probe_resnet101, 0.01),
        ("Full Fine-tune",  build_finetune_resnet101,    0.01),
        ("LoRA-C (r=2)",    lambda: build_lora_c_resnet101(rank=4, alpha=None, unfreeze_bn=False), 0.01),
    ]

    for name, builder, lr in configs:
        model = builder().to(device)
        # trying paper recommendation
        optimizer = optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, momentum=0.9, weight_decay=5e-4
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

        best_test = 0.0
        print(f"\n  {name}:")
        for ep in range(1, N_EPOCHS + 1):
            tr_loss, tr_acc = train_one_epoch(model, sub_loader, optimizer, criterion)
            te_loss, te_acc = evaluate(model, test_loader, criterion)
            scheduler.step()
            best_test = max(best_test, te_acc)
            gap = tr_acc - te_acc
            print(f"    Ep {ep:2d}/{N_EPOCHS} | "
                  f"Train Loss {tr_loss:.3f} Acc {tr_acc:5.1f}% | "
                  f"Test Loss {te_loss:.3f} Acc {te_acc:5.1f}% | "
                  f"Gap {gap:+5.1f}%")

        trainable, total = count_trainable(model)
        pct = 100 * trainable / total
        print(f"  → {name} Best test: {best_test:.1f}% | Params: {trainable:,} ({pct:.2f}%)")

        limited_results.append({
            'fraction': f"{frac*100:.0f}%",
            'n_images': n_images,
            'method': name,
            'best_test': round(best_test, 1)
        })
        del model
        torch.cuda.empty_cache()

# quick summary
ld_df = pd.DataFrame(limited_results)
pivot = ld_df.pivot(index='fraction', columns='method', values='best_test')
print(f"\n{'='*60}")
print("  LIMITED DATA RESULTS")
print(f"{'='*60}")
print(pivot.to_string())

print("\n  LoRA-C compared to full FT:")
for _, row in ld_df[ld_df['method'] == 'LoRA-C (r=4)'].iterrows():
    ft_row = ld_df[(ld_df['fraction'] == row['fraction']) &
                    (ld_df['method'] == 'Full Fine-tune')]
    if len(ft_row) > 0:
        gap = row['best_test'] - ft_row.iloc[0]['best_test']
        print(f"    {row['fraction']:>5s} data: LoRA-C {row['best_test']:.1f}% vs FT {ft_row.iloc[0]['best_test']:.1f}% → gap {gap:+.1f}%")

In [ ]:
# Limited data experiment ( r=2, BN frozen )
# same as above but seeing the difference if BN properly frozen using a different train epoch function
from torch.utils.data import Subset
import numpy as np
import pandas as pd

N_EPOCHS = 15
criterion = nn.CrossEntropyLoss()

def get_stratified_subset(dataset, fraction, seed=42):
    np.random.seed(seed)
    targets = np.array([dataset[i][1] for i in range(len(dataset))])
    indices = []
    for c in np.unique(targets):
        c_idx = np.where(targets == c)[0]
        n_keep = max(1, int(len(c_idx) * fraction))
        indices.extend(np.random.choice(c_idx, n_keep, replace=False))
    return Subset(dataset, indices)

fractions = [0.1, 0.25, 0.5, 1.0]
limited_results = []

for frac in fractions:
    subset = get_stratified_subset(train_set, frac)
    sub_loader = DataLoader(subset, batch_size=64, shuffle=True,
                            num_workers=2, pin_memory=True)
    n_images = len(subset)
    print(f"\n{'='*60}")
    print(f"  Data: {frac*100:.0f}% ({n_images} images)")
    print(f"{'='*60}")

    configs = [
        ("Linear Probe",   build_linear_probe_resnet101, 0.01),
        ("Full Fine-tune",  build_finetune_resnet101,    0.01),
        ("LoRA-C (r=2)",    lambda: build_lora_c_resnet101(rank=2, alpha=None, unfreeze_bn=False), 0.01),
    ]

    for name, builder, lr in configs:
        model = builder().to(device)
        #
        optimizer = optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, momentum=0.9, weight_decay=5e-4
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

        best_test = 0.0
        print(f"\n  {name}:")
        for ep in range(1, N_EPOCHS + 1):
            # keep BN unfrozen for full fine tune
            if name == "Full Fine-tune":
                tr_loss, tr_acc = train_one_epoch(model, sub_loader, optimizer, criterion)
            else:
                tr_loss, tr_acc = train_one_epoch_with_frozen_bn(model, sub_loader, optimizer, criterion)

            te_loss, te_acc = evaluate(model, test_loader, criterion)
            scheduler.step()
            best_test = max(best_test, te_acc)
            gap = tr_acc - te_acc
            print(f"    Ep {ep:2d}/{N_EPOCHS} | "
                  f"Train Loss {tr_loss:.3f} Acc {tr_acc:5.1f}% | "
                  f"Test Loss {te_loss:.3f} Acc {te_acc:5.1f}% | "
                  f"Gap {gap:+5.1f}%")

        trainable, total = count_trainable(model)
        pct = 100 * trainable / total
        print(f"  → {name} Best test: {best_test:.1f}% | Params: {trainable:,} ({pct:.2f}%)")

        limited_results.append({
            'fraction': f"{frac*100:.0f}%",
            'n_images': n_images,
            'method': name,
            'best_test': round(best_test, 1)
        })
        del model
        torch.cuda.empty_cache()

# quick summary
ld_df = pd.DataFrame(limited_results)
pivot = ld_df.pivot(index='fraction', columns='method', values='best_test')
print(f"\n{'='*60}")
print("  LIMITED DATA RESULTS")
print(f"{'='*60}")
print(pivot.to_string())

print("\n  LoRA-C advantage over Full FT:")
for _, row in ld_df[ld_df['method'] == 'LoRA-C (r=2)'].iterrows():
    ft_row = ld_df[(ld_df['fraction'] == row['fraction']) &
                    (ld_df['method'] == 'Full Fine-tune')]
    if len(ft_row) > 0:
        gap = row['best_test'] - ft_row.iloc[0]['best_test']
        print(f"    {row['fraction']:>5s} data: LoRA-C {row['best_test']:.1f}% vs FT {ft_row.iloc[0]['best_test']:.1f}% → gap {gap:+.1f}%")

In [ ]:
# freeze BN entirely or only parameters, LoRA only r=2


from torch.utils.data import Subset
import numpy as np
import pandas as pd


def get_stratified_subset(dataset, fraction, seed=42):
    np.random.seed(seed)
    targets = np.array([dataset[i][1] for i in range(len(dataset))])
    indices = []
    for c in np.unique(targets):
        c_idx = np.where(targets == c)[0]
        n_keep = max(1, int(len(c_idx) * fraction))
        indices.extend(np.random.choice(c_idx, n_keep, replace=False))
    return Subset(dataset, indices)


N_EPOCHS_BN = 10
criterion = nn.CrossEntropyLoss()

subset = get_stratified_subset(train_set, 0.25, seed=42)
sub_loader = DataLoader(subset, batch_size=64, shuffle=True,
                        num_workers=2, pin_memory=True)

bn_results = []

configs = [
    ("LoRA-C r=2, BN stats updating", train_one_epoch),
    ("LoRA-C r=2, BN stats frozen", train_one_epoch_with_frozen_bn),
]


for name, train_fn in configs:
    print(f"\n{'='*70}\n{name}\n{'='*70}")

    model = build_lora_c_resnet101(rank=2, alpha=None, unfreeze_bn=False).to(device)

    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.01, momentum=0.9, weight_decay=5e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_BN)

    best_test = 0.0

    for ep in range(1, N_EPOCHS_BN + 1):
        tr_loss, tr_acc = train_fn(model, sub_loader, optimizer, criterion)
        te_loss, te_acc = evaluate(model, test_loader, criterion)
        scheduler.step()

        best_test = max(best_test, te_acc)
        print(f"Ep {ep:2d}/{N_EPOCHS_BN} | Train {tr_acc:5.1f}% | Test {te_acc:5.1f}% | Best {best_test:5.1f}%")

    trainable, total = count_trainable(model)

    bn_results.append({
        "setting": name,
        "data": "25%",
        "best_test": round(best_test, 1),
        "trainable_params": trainable,
        "pct_trainable": round(100 * trainable / total, 2)
    })

    del model
    torch.cuda.empty_cache()

bn_df = pd.DataFrame(bn_results)
print(bn_df.to_string(index=False))

In [ ]:
# Compare LoRA-C with frozen BN vs unfrozen BN

from torch.utils.data import Subset
import numpy as np
import pandas as pd

def get_stratified_subset(dataset, fraction, seed=42):
    np.random.seed(seed)
    targets = np.array([dataset[i][1] for i in range(len(dataset))])
    indices = []

    for c in np.unique(targets):
        c_idx = np.where(targets == c)[0]
        n_keep = max(1, int(len(c_idx) * fraction))
        indices.extend(np.random.choice(c_idx, n_keep, replace=False))

    return Subset(dataset, indices)


N_EPOCHS_BN = 10
criterion = nn.CrossEntropyLoss()

subset = get_stratified_subset(train_set, 0.25, seed=42)
sub_loader = DataLoader(
    subset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

bn_results = []

configs = [
    {
        "name": "LoRA-C r=2, BN frozen",
        "unfreeze_bn": False,
        "train_fn": train_one_epoch_with_frozen_bn,
    },
    {
        "name": "LoRA-C r=2, BN unfrozen",
        "unfreeze_bn": True,
        "train_fn": train_one_epoch,
    },
]

for cfg in configs:
    name = cfg["name"]
    unfreeze_bn = cfg["unfreeze_bn"]
    train_fn = cfg["train_fn"]

    print(f"\n{'='*70}")
    print(name)
    print(f"{'='*70}")

    model = build_lora_c_resnet101(
        rank=2,
        alpha=None,
        unfreeze_bn=unfreeze_bn
    ).to(device)

    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.01,
        momentum=0.9,
        weight_decay=5e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=N_EPOCHS_BN
    )

    best_test = 0.0

    for ep in range(1, N_EPOCHS_BN + 1):
        tr_loss, tr_acc = train_fn(model, sub_loader, optimizer, criterion)
        te_loss, te_acc = evaluate(model, test_loader, criterion)

        scheduler.step()
        best_test = max(best_test, te_acc)

        print(
            f"Ep {ep:2d}/{N_EPOCHS_BN} | "
            f"Train {tr_acc:5.1f}% | "
            f"Test {te_acc:5.1f}% | "
            f"Best {best_test:5.1f}%"
        )

    trainable, total = count_trainable(model)

    bn_results.append({
        "setting": name,
        "data": "25%",
        "best_test": round(best_test, 1),
        "trainable_params": trainable,
        "pct_trainable": round(100 * trainable / total, 2)
    })

    del model
    torch.cuda.empty_cache()

bn_df = pd.DataFrame(bn_results)
print("\nBN comparison results:")
print(bn_df.to_string(index=False))


LoRA-C r=2, BN frozen
LoRA-C ResNet-101 (r=2, α=4.0) | Trainable: 352,741 / 42,843,493 (0.82%)
Ep  1/10 | Train  36.9% | Test  82.8% | Best  82.8%
Ep  2/10 | Train  88.5% | Test  87.8% | Best  87.8%
Ep  3/10 | Train  93.5% | Test  88.6% | Best  88.6%
Ep  4/10 | Train  94.7% | Test  85.8% | Best  88.6%
Ep  5/10 | Train  93.6% | Test  86.6% | Best  88.6%
Ep  6/10 | Train  98.0% | Test  89.3% | Best  89.3%
Ep  7/10 | Train  98.3% | Test  90.0% | Best  90.0%
Ep  8/10 | Train  99.0% | Test  90.0% | Best  90.0%
Ep  9/10 | Train  99.5% | Test  90.0% | Best  90.0%
Ep 10/10 | Train  99.6% | Test  90.0% | Best  90.0%

LoRA-C r=2, BN unfrozen
LoRA-C ResNet-101 (r=2, α=4.0) | Trainable: 458,085 / 42,843,493 (1.07%)
Ep  1/10 | Train  29.8% | Test  80.4% | Best  80.4%
Ep  2/10 | Train  84.1% | Test  85.6% | Best  85.6%
Ep  3/10 | Train  93.6% | Test  86.0% | Best  86.0%
Ep  4/10 | Train  95.1% | Test  86.7% | Best  86.7%
Ep  5/10 | Train  96.5% | Test  87.4% | Best  87.4%
Ep  6/10 | Train  96.8% | 

In [ ]:
# compare rank settings in LoRA

import pandas as pd

N_EPOCHS = 10
criterion = nn.CrossEntropyLoss()

ranks = [1, 2, 4, 8, 16, 32]
rank_results = []
rank_curves = []


for r in ranks:
    alpha = 2.0 * r
    model = build_lora_c_resnet101(rank=r, alpha=alpha, unfreeze_bn=False).to(device)

    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.01, momentum=0.9, weight_decay=5e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

    best_test = 0.0
    for ep in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        _, te_acc = evaluate(model, test_loader, criterion)
        scheduler.step()
        best_test = max(best_test, te_acc)

        rank_curves.append({
            'rank': r, 'epoch': ep,
            'train_acc': tr_acc, 'test_acc': te_acc
        })

        gap = tr_acc - te_acc
        print(f"  r={r:2d} | Ep {ep:2d}/{N_EPOCHS} | "
              f"Train {tr_acc:5.1f}% | Test {te_acc:5.1f}% | Gap {gap:+5.1f}%")

    trainable, total = count_trainable(model)
    pct = 100 * trainable / total
    print(f"  r={r:2d} → Best test: {best_test:.1f}% | Params: {trainable:,} ({pct:.2f}%)")
    print()

    rank_results.append({
        'rank': r, 'alpha': alpha,
        'best_test': round(best_test, 1),
        'trainable': trainable,
        'pct_trainable': round(pct, 2),
        'bn': 'unfrozen'
    })
    del model
    torch.cuda.empty_cache()

rank_df = pd.DataFrame(rank_results)
rank_curves_df = pd.DataFrame(rank_curves)
print("\n" + rank_df.to_string(index=False))